# Solutions · 6. Spectra, the Beer–Lambert law and preprocessing

Worked solutions to the three exercises of [Lecture 6](../../notebooks/part3_chemometrics/06_beer_lambert_spectroscopy_and_spectral_preprocessing.ipynb). Each solution is computed here and checked with `assert` statements against an exact identity or an independent calculation. The first code cell sets up the notebook and repeats the demonstrations of the lecture without printing them, because the solutions use their variables.

In [1]:
try:                      # use the installed package, or the copy in this repository
    import engml_2_applied
except ImportError:
    import pathlib, subprocess, sys
    for _root in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_root / "engml_2_applied" / "__init__.py").exists():
            sys.path.insert(0, str(_root))
            break
    else:                 # on Google Colab: install from GitHub
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "git+https://github.com/ktwyw/engml-2-applied.git"], check=True)
    import engml_2_applied

import inspect
import numpy as np
import matplotlib.pyplot as plt
from engml_2_applied import (clustering_phases, constants, descriptors, dim_reduction, linear_physics,
                             trees_ensembles, uncertainty)
from engml_2_applied.plotting import panel_label, use_style

use_style()

def show_source(obj):
    """Return the source code of a library function or class."""
    return inspect.getsource(obj)

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

import io, sys
_stdout, sys.stdout = sys.stdout, io.StringIO()   # the demonstrations are repeated without their output
heading("Demonstration 1: Exact Path-Length & Offset Invariance of Standard Normal Variate (SNV)")
wl = np.linspace(1100.0, 1700.0, 120)
pure_peak1 = np.exp(-0.5 * ((wl - 1320.0) / 45.0) ** 2)
pure_peak2 = np.exp(-0.5 * ((wl - 1480.0) / 50.0) ** 2)
base_spectrum = 0.6 * pure_peak1 + 0.4 * pure_peak2

# Create 10 spectra with varying multiplicative scatter b_i in [0.5, 2.5] and additive offset alpha_i in [-1, 3]:
b_scales = np.linspace(0.5, 2.5, 10)[:, None]
a_offsets = np.linspace(-1.0, 3.0, 10)[:, None]
X_scattered = b_scales * base_spectrum[None, :] + a_offsets

X_snv = dim_reduction.snv_preprocess(X_scattered)
raw_spread = float(np.max(np.abs(X_scattered - X_scattered[0:1, :])))
max_snv_spread = float(np.max(np.abs(X_snv - X_snv[0:1, :])))
print(f"Max pairwise difference across 10 distorted spectra before snv: {raw_spread:.4f} AU")
print(f"Max pairwise difference across 10 distorted spectra AFTER  SNV: {max_snv_spread:.2e} AU")
assert np.allclose(np.mean(X_snv, axis=1), 0.0, atol=1e-14)
assert np.allclose(np.std(X_snv, axis=1), 1.0, atol=1e-14)
assert max_snv_spread < 1e-13
heading("Demonstration 2: Spectral First Derivative + SNV Annihilation of Linear Baseline Tilt")
d_wl = float(wl[1] - wl[0])
X_offset_only = base_spectrum[None, :] + a_offsets
dX_dlambda = dim_reduction.spectral_first_derivative(X_offset_only, d_lambda=d_wl)
max_deriv_spread = float(np.max(np.abs(dX_dlambda - dX_dlambda[0:1, :])))
print(f"Max spread after 1st spectral derivative on offset spectra : {max_deriv_spread:.2e}")
assert max_deriv_spread < 1e-13

# Now combine multiplicative scatter b_i, constant offset a_i, AND linear tilt beta_i * (wl - 1400):
beta_tilts = np.linspace(-0.0015, 0.0015, 10)[:, None]
X_tilted_scattered = b_scales * base_spectrum[None, :] + a_offsets + (b_scales * 0.0005) * (wl[None, :] - 1400.0)
X_d1_snv = dim_reduction.snv_preprocess(dim_reduction.spectral_first_derivative(X_tilted_scattered, d_lambda=d_wl))
print(f"Max spread after (1st Derivative -> SNV) on tilted spectra: {np.max(np.abs(X_d1_snv - X_d1_snv[0:1, :])):.2e}")
assert np.max(np.abs(X_d1_snv - X_d1_snv[0:1, :])) < 1e-13
heading("Demonstration 3: Soft-Sensor Calibration Benchmark — Raw vs. SNV Preprocessed NIR")
rng = np.random.default_rng(42)
N_cal = 50
c1 = rng.uniform(0.1, 0.9, size=N_cal)
scatter_b = rng.uniform(0.5, 2.0, size=(N_cal, 1))
offset_a = rng.uniform(-0.8, 0.8, size=(N_cal, 1))
# Analyte peak1 on top of a constant reference background matrix peak2:
X_raw_nir = scatter_b * (np.outer(c1, pure_peak1) + pure_peak2[None, :]) + offset_a + rng.normal(0.0, 0.002, size=(N_cal, len(wl)))

pls_raw = dim_reduction.pls1_nipals(X_raw_nir, c1, n_components=2)
pls_snv = dim_reduction.pls1_nipals(dim_reduction.snv_preprocess(X_raw_nir), c1, n_components=2)
rmse_raw = float(np.sqrt(np.mean((pls_raw["y_pred"] - c1) ** 2)))
rmse_snv = float(np.sqrt(np.mean((pls_snv["y_pred"] - c1) ** 2)))
print(f"2-Component PLS1 RMSE: Raw Scattered Spectra = {rmse_raw:.4f} mol/L -> SNV Preprocessed = {rmse_snv:.4f} mol/L")
assert rmse_snv < 0.35 * rmse_raw
heading("Demonstration 4: Detector Noise vs. Derivative Amplification Tradeoff")
print(f"{'Detector Noise [AU]':>19s} | {'Raw PLS1 RMSE':>14s} | {'SNV PLS1 RMSE':>14s} | {'Deriv+SNV RMSE':>15s}")
print("-" * 70)
for sig_det in [1e-4, 1e-3, 5e-3, 2e-2]:
    X_n = scatter_b * (np.outer(c1, pure_peak1) + pure_peak2[None, :]) + offset_a + rng.normal(0.0, sig_det, size=(N_cal, len(wl)))
    r_raw = float(np.sqrt(np.mean((dim_reduction.pls1_nipals(X_n, c1, 2)["y_pred"] - c1) ** 2)))
    r_snv = float(np.sqrt(np.mean((dim_reduction.pls1_nipals(dim_reduction.snv_preprocess(X_n), c1, 2)["y_pred"] - c1) ** 2)))
    X_ds = dim_reduction.snv_preprocess(dim_reduction.spectral_first_derivative(X_n, d_lambda=d_wl))
    r_ds = float(np.sqrt(np.mean((dim_reduction.pls1_nipals(X_ds, c1, 2)["y_pred"] - c1) ** 2)))
    print(f"{sig_det:19.4f} | {r_raw:14.4f} | {r_snv:14.4f} | {r_ds:15.4f}")
heading("Inside the Algorithm: By-Hand Row-Wise SNV vs. the Library")
mu_row = np.mean(X_scattered, axis=1, keepdims=True)
std_row = np.std(X_scattered, axis=1, keepdims=True)
X_snv_hand = (X_scattered - mu_row) / std_row
diff_snv = float(np.max(np.abs(X_snv_hand - X_snv)))
print("by hand (pure NumPy) row 0[:5]:", np.round(X_snv_hand[0, :5], 6))
print("library (engml_2_applied)      row 0[:5]:", np.round(X_snv[0, :5], 6))
print(f"Max discrepancy               : {diff_snv:.2e}")
assert diff_snv < 1e-15
sys.stdout = _stdout
print("Set-up complete: the variables of the lecture's demonstrations are defined.")

Set-up complete: the variables of the lecture's demonstrations are defined.


## Exercise 1

**Problem.** Corrupt the binary mixture spectra `X_raw_nir` with linear baseline tilt $\beta_i (\lambda - 1400)$ ($\beta_i \in [-0.001, 0.001]$). Apply `spectral_first_derivative` followed by `snv_preprocess`, fit a 2-component `pls1_nipals` model predicting $c_1$, and verify that preprocessing reduces calibration RMSE by more than $3\times$ compared to raw spectra.

In [2]:
heading("Exercise 1: Combined Linear Baseline Tilt + Scatter Removal via Derivative + SNV")
tilt_beta = rng.uniform(-0.001, 0.001, size=(N_cal, 1))
X_drift = X_raw_nir + tilt_beta * (wl[None, :] - 1400.0)
X_prep = dim_reduction.snv_preprocess(dim_reduction.spectral_first_derivative(X_drift, d_lambda=d_wl))

rmse_drift_raw = float(np.sqrt(np.mean((dim_reduction.pls1_nipals(X_drift, c1, n_components=2)["y_pred"] - c1) ** 2)))
rmse_drift_prep = float(np.sqrt(np.mean((dim_reduction.pls1_nipals(X_prep, c1, n_components=2)["y_pred"] - c1) ** 2)))
print(f"Tilted+Scattered RMSE: Raw = {rmse_drift_raw:.4f} -> Derivative+SNV = {rmse_drift_prep:.4f}")
assert rmse_drift_prep < rmse_drift_raw / 3.0


Exercise 1: Combined Linear Baseline Tilt + Scatter Removal via Derivative + SNV
Tilted+Scattered RMSE: Raw = 0.1030 -> Derivative+SNV = 0.0047


**Discussion.** When both linear tilt and multiplicative scatter are present, chaining `spectral_first_derivative` $\to$ `snv_preprocess` reduces the calibration error from $0.103$ to $0.0047\;\mathrm{mol/L}$.

---

## Exercise 2

**Problem.** Explain and verify why SNV fails on a **single-analyte pure Beer–Lambert solution** without an internal reference peak or solvent background ($A_i(\lambda_j) = b_i c_i \varepsilon_1(\lambda_j) + \alpha_i$): show that SNV divides out $b_i c_i$ completely, collapsing all concentrations $c_i \in [0.1, 2.0]\;\mathrm{mol/L}$ onto the exact same normalized spectrum $\varepsilon_1^{\mathrm{SNV}}(\lambda)$.

In [3]:
heading("Exercise 2: Single-Chromophore SNV Self-Cancellation Diagnostic Audit")
c_single = np.linspace(0.1, 2.0, 15)[:, None]
X_single_chrom = c_single * pure_peak1[None, :]
X_single_snv = dim_reduction.snv_preprocess(X_single_chrom)
spread_single = float(np.max(np.abs(X_single_snv - X_single_snv[0:1, :])))
print(f"Max spectral difference across 15 concentrations after SNV on single peak: {spread_single:.2e}")
assert spread_single < 1e-14


Exercise 2: Single-Chromophore SNV Self-Cancellation Diagnostic Audit
Max spectral difference across 15 concentrations after SNV on single peak: 8.88e-16


**Discussion.** Because $c_i$ enters $A_i(\lambda_j) = (b_i c_i)\varepsilon_1(\lambda_j)$ purely as an overall scale factor when only one absorbing species is present, SNV cancels $c_i$ along with $b_i$. Quantitative SNV requires either an internal reference band (such as a solvent peak) or multiple varying species so that relative peak ratios encode concentration.

---

## Exercise 3

**Problem.** Implement **multiplicative scatter correction (MSC, Geladi et al. 1985)** `msc_preprocess(X: np.ndarray, ref_spectrum: np.ndarray | None = None) -> np.ndarray`, which regresses each row spectrum $\mathbf{x}_i$ onto a reference spectrum $\mathbf{x}_{\mathrm{ref}}$ (defaulting to the column mean $\bar{\mathbf{x}}$) via simple linear regression $x_{i,j} \approx a_i + b_i x_{\mathrm{ref},j}$ and returns the scatter-corrected spectrum $(x_{i,j} - \hat{a}_i)/\hat{b}_i$. Verify on `X_scattered` that MSC recovers `base_spectrum` to $<10^{-13}$.

In [4]:
heading("Exercise 3: Implement Multiplicative Scatter Correction (MSC) From Scratch")
def msc_preprocess(X: np.ndarray, ref_spectrum: np.ndarray | None = None) -> np.ndarray:
    """Multiplicative Scatter Correction (MSC) aligning each spectrum to ref_spectrum via OLS."""
    X_mat = np.atleast_2d(np.asarray(X, dtype=float))
    ref = np.mean(X_mat, axis=0) if ref_spectrum is None else np.asarray(ref_spectrum, dtype=float).ravel()
    ref_mean = float(np.mean(ref))
    ref_c = ref - ref_mean
    ref_var = float(np.dot(ref_c, ref_c))
    row_means = np.mean(X_mat, axis=1, keepdims=True)
    b_hat = ((X_mat - row_means) @ ref_c)[:, None] / max(ref_var, 1e-15)
    a_hat = row_means - b_hat * ref_mean
    return (X_mat - a_hat) / np.where(np.abs(b_hat) < 1e-15, 1.0, b_hat)

X_msc = msc_preprocess(X_scattered, ref_spectrum=base_spectrum)
msc_err = float(np.max(np.abs(X_msc - base_spectrum[None, :])))
print(f"Max reconstruction error of MSC against true base_spectrum: {msc_err:.2e} AU")
assert msc_err < 1e-13


Exercise 3: Implement Multiplicative Scatter Correction (MSC) From Scratch
Max reconstruction error of MSC against true base_spectrum: 3.40e-16 AU


**Discussion.** SNV and MSC are algebraically linked linear transformations ($A^{\mathrm{MSC}}$ is an affine rescaling of $A^{\mathrm{SNV}}$); SNV requires no stored reference spectrum, whereas MSC preserves the original physical absorbance units ($\mathrm{AU}$) of the reference spectrum.